In [ ]:
#Objective: To build and evaluate ML models to predict customer churn using telco customer churn dataset.
#Steps include: preprocessing, encoding, applying feature engineering to improve predictions and avoid redundancy.
#Training two models: random forest classifier and logistic regression
#Evalating model performance using classigication metrics, confusion matrix and ROC-AUC
#from google.colab import drive
#drive.mount('/content/drive')
#from google.colab import files
#uploaded = files.upload()
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")
#df = pd.read_csv('Telco-Customer-Churn-rawdata.csv')
df = df.replace(r'^\s*$', np.nan, regex=True)# replace blank space with NaN
df = df.drop(['customerID'], axis = 1)
df['TotalCharges'] = pd.to_numeric(df.TotalCharges, errors='coerce')
df[np.isnan(df['TotalCharges'])]
df.drop(labels=df[df['tenure'] == 0].index, axis=0, inplace=True)
df['TotalCharges'].fillna(df['TotalCharges'].mean(), inplace=True)
df["SeniorCitizen"]= df["SeniorCitizen"].map({0: "No", 1: "Yes"})
#encoding categorical values
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
#feature engineering
df['AvgCharges'] = df['TotalCharges'] / df['tenure']
df['AvgCharges'] = df['AvgCharges'].fillna(0)
services = ['PhoneService', 'MultipleLines',
            'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies']

df['NumServices'] = df[services].apply(lambda x: sum(x == 'Yes'), axis=1)
protection = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
df['ProtectionCount'] = df[protection].apply(lambda x: sum(x == 'Yes'), axis=1)

df = pd.get_dummies(df, drop_first = True)#converts categorical values to numerical

#split data
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
#logistic regression
#scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)#fit on training data
X_test_scaled = scaler.transform(X_test)
#train
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train_scaled, y_train)
#predict
y_pred_lr = model_lr.predict(X_test_scaled)
print("\nLOGISTIC REGRESSION")
print(classification_report(y_test, y_pred_lr))
print("Confusion Matrix(logistic regression): \n", confusion_matrix(y_test, y_pred_lr))
y_prob_lr = model_lr.predict_proba(X_test_scaled)[:, 1]
print("ROC-AUC (Logistic):", roc_auc_score(y_test, y_prob_lr))

#Random forest
#train
model_rf = RandomForestClassifier(n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    random_state=42)
model_rf.fit(X_train, y_train)
#predict
y_pred_rf = model_rf.predict(X_test)
print("\nRANDOM FOREST")
print(classification_report(y_test, y_pred_rf))
cm = confusion_matrix(y_test, y_pred_rf)
print("Confusion Matrix(random forest): \n",cm)
y_prob_rf = model_rf.predict_proba(X_test)[:, 1]
print("ROC-AUC (Random Forest):", roc_auc_score(y_test, y_prob_rf))

#key insights
#1. 1. Overall Model Performance
# Both Logistic Regression and Random Forest achieved similar performance:
#F1-score (churn): ~0.56
#ROC-AUC: ~0.83
#This indicates both models are equally capable of distinguishing between churn and non-churn customers.

#2. Recall vs Precision Trade-off
#Logistic Regression achieved higher recall for churn (0.51 vs 0.48), it correctly identified more customers who churned.
#Random Forest achieved higher precision (0.66 vs 0.62), it made fewer false churn predictions.
#3. Confusion Matrix Insights
#Logistic Regression correctly identified 192 churn customers, while Random Forest identified 180.
#Random Forest produced fewer false positives, but more false negatives.
#4. Impact of Feature Engineering
#Feature engineering had limited impact on Random Forest performance whereas, it slightly improved recall in Logistic Regression, suggesting engineered features are more beneficial for linear models.

#5. Final Conclusion
#Both models perform similarly overall, but Logistic Regression is better suited for churn prediction due to its higher ability to detect at-risk customers.
#Future improvements can focus on:
#Hyperparameter tuning
#Threshold adjustment to further improve recall